In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, roc_auc_score, f1_score, roc_curve, classification_report, 
                             cohen_kappa_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import warnings
from sklearn.exceptions import ConvergenceWarning
import seaborn as sns
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

from sklearn.svm import SVC


#suppress convergence warnings
warnings.simplefilter("ignore", ConvergenceWarning)
pd.options.mode.chained_assignment = None

In [11]:
cannabis = pd.read_csv("cannabis_full.csv")
cannabis.dropna(inplace = True)

# Part One: Binary Classification

Create a dataset that is limited only to the Sativa and Indica type cannabis strains.


In [13]:
# filter dataset
filtered_data = cannabis[(cannabis["Type"] == "sativa") | (cannabis["Type"] == "indica")]
filtered_data = filtered_data.drop(["Effects", "Flavor"], axis = 1)

This section asks you to create a final best model for each of the four new model types studied this week: LDA, QDA, SVC, and SVM. For SVM, you may limit yourself to only the polynomial kernel.

For each, you should:

Choose a metric you will use to select your model, and briefly justify your choice. (Hint: There is no specific target category here, so this should not be a metric that only prioritizes one category.)

* Find the best model for predicting the Type variable. Don’t forget to tune any hyperparameters.

* Report the (cross-validated!) metric.

* Fit the final model.

* Output a confusion matrix.

## LDA

In [67]:
X = filtered_data.drop(["Type", "Strain"], axis = 1)
y = filtered_data["Type"]

# make y dummy vars
y = y.map({'sativa': 0, 'indica': 1}).astype(float)

filtered_data['Type'].value_counts()

Type
indica    659
sativa    409
Name: count, dtype: int64

In [76]:

def get_metrics(X, y, model_type):
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify = y)

    # Initialize model and params
    if model_type == "LDA":
        model = LinearDiscriminantAnalysis()
        params = {"model__solver": ['svd', 'lsqr']}
    elif model_type == "QDA":
        model = QuadraticDiscriminantAnalysis()
        params = {'model__reg_param': [0.0, 0.1, 0.2, 0.5, 1.0]}
    elif model_type == "SVC":
        model = SVC()
        params = {'model__kernel': ['linear', 'poly', 'rbf'], 'model__C': [0.1, 1, 10], 
                  'model__gamma': ['scale', 'auto']}
    elif model_type == "SVM":
        model = SVC()
        params = {'model__kernel': ['poly'], 'model__C': [0.1, 1, 10], 'model__degree': [3, 4, 5]}

    # Define preprocessing pipeline
    ct = ColumnTransformer(
        transformers=[('scaler', StandardScaler(), ["Rating"])],
        remainder = "passthrough"
    )

    # Build pipeline with column transformer and model
    pipeline = Pipeline([
        ('preprocessor', ct),
        ('model', model)
    ])

    # Perform Grid Search for hyperparameter tuning
    grid_search = GridSearchCV(pipeline, param_grid=params, cv=5, scoring='f1_macro', n_jobs = -1)
    grid_search.fit(X_train, y_train)

    # Get the best model and best parameters
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    print(f"Best Model: {best_params}")

    # Cross-validation to get F1 score
    cv_f1 = cross_val_score(best_model, X_train, y_train, cv=5, scoring='f1_macro').mean()
    print(f"Cross-validated F1 score: {cv_f1:.4f}")

    # Fit the final model on the whole training set
    best_model.fit(X_train, y_train)

    # Predict on the test set and calculate confusion matrix
    y_pred = best_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:")
    print(cm)

    return



In [77]:
# calculate for LDA
get_metrics(X, y, model_type = "LDA")

Best Model: {'model__solver': 'lsqr'}
Best Model: {'model__solver': 'lsqr'}


Cross-validated F1 score: 0.8532
Confusion Matrix:
[[ 60  22]
 [ 21 111]]
Cross-validated F1 score: 0.8532
Confusion Matrix:
[[ 60  22]
 [ 21 111]]


## QDA

In [78]:
get_metrics(X, y, model_type = "QDA")

/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonPro

Best Model: {'model__reg_param': 0.1}
Cross-validated F1 score: 0.8532
Best Model: {'model__reg_param': 0.1}
Cross-validated F1 score: 0.8532


Confusion Matrix:
[[ 66  16]
 [ 17 115]]
Confusion Matrix:
[[ 66  16]
 [ 17 115]]


/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
/Users/chloefeehan/.conda/envs/pythonProject/lib/python3.10/site-packages/sklearn/discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


## SVC

In [79]:
get_metrics(X, y, model_type = "SVC")

Best Model: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Cross-validated F1 score: 0.8630
Confusion Matrix:
[[ 67  15]
 [ 20 112]]
Best Model: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'linear'}
Cross-validated F1 score: 0.8630
Confusion Matrix:
[[ 67  15]
 [ 20 112]]


## SVM

In [80]:
get_metrics(X, y, model_type = "SVM")

Best Model: {'model__C': 1, 'model__degree': 3, 'model__kernel': 'poly'}
Cross-validated F1 score: 0.8571
Confusion Matrix:
[[ 64  18]
 [ 18 114]]
Best Model: {'model__C': 1, 'model__degree': 3, 'model__kernel': 'poly'}
Cross-validated F1 score: 0.8571
Confusion Matrix:
[[ 64  18]
 [ 18 114]]
